In [49]:
import pandas as pd
import numpy as np

In [50]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)

In [51]:
dealer_df = pd.read_csv("raw_data/dealer.csv")
dealer_df.head()

,CERTIFICATE-NUMBER,OWNERSHIP,CERTIFICATE-DATE,EXPIRATION-DATE,EXPIRATION-FLAG,CERTIFICATE-ISSUE-COUNT,NAME,STREET,STREET2,CITY,STATE-ABBREV,ZIP-CODE,OTHER-NAMES-COUNT,OTHER-NAMES-1,OTHER-NAMES-2,OTHER-NAMES-3,OTHER-NAMES-4,OTHER-NAMES-5,OTHER-NAMES-6,OTHER-NAMES-7,OTHER-NAMES-8,OTHER-NAMES-9,OTHER-NAMES-10,OTHER-NAMES-11,OTHER-NAMES-12,OTHER-NAMES-13,OTHER-NAMES-14,OTHER-NAMES-15,OTHER-NAMES-16,OTHER-NAMES-17,OTHER-NAMES-18,OTHER-NAMES-19,OTHER-NAMES-20,OTHER-NAMES-21,OTHER-NAMES-22,OTHER-NAMES-23,OTHER-NAMES-24,OTHER-NAMES-25,Unnamed: 38
0,00-0027,3,20000104,20010103,*,0,BIRD-EARLE JOINT VENTURES INC,4900 US HIGHWAY 1 N STE 300,,SAINT AUGUSTINE,FL,320956265,0,,,,,,,,,,,,,,,,,,,,,,,,,,NaN
1,00-0037,7,20000105,20010104,*,0,FLY AND BE FREE LLC,32 WALKER DR,,SIMSBURY,CT,060702624,0,,,,,,,,,,,,,,,,,,,,,,,,,,NaN
2,00-0040,1,20000105,20010104,*,0,CALL O JAY,83 POOL CREEK RD,,LIVINGSTON,MT,590479117,0,,,,,,,,,,,,,,,,,,,,,,,,,,NaN
3,00-0056,3,20000112,20010111,*,0,MAC AIR CORP,PO BOX 189,,MCPHERSON,KS,674600189,0,,,,,,,,,,,,,,,,,,,,,,,,,,NaN
4,00-0060,1,20000107,20010106,*,0,CONNOLLY WILLIAM DBA,2520 TAILSPIN TRAIL,,DAYTONA BEACH,FL,32124,1,SHAMROCK AIR-1,,,,,,,,,,,,,,,,,,,,,,,,,NaN


In [52]:
dealer_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 12400 entries, 0 to 12399
Data columns (total 39 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   CERTIFICATE-NUMBER       12400 non-null  str    
 1   OWNERSHIP                12400 non-null  int64  
 2   CERTIFICATE-DATE         12400 non-null  str    
 3   EXPIRATION-DATE          12400 non-null  str    
 4   EXPIRATION-FLAG          12400 non-null  str    
 5   CERTIFICATE-ISSUE-COUNT  12400 non-null  int64  
 6   NAME                     12400 non-null  str    
 7   STREET                   12400 non-null  str    
 8   STREET2                  12400 non-null  str    
 9   CITY                     12400 non-null  str    
 10  STATE-ABBREV             12400 non-null  str    
 11  ZIP-CODE                 12400 non-null  str    
 12  OTHER-NAMES-COUNT        12400 non-null  int64  
 13  OTHER-NAMES-1            12400 non-null  str    
 14  OTHER-NAMES-2            12400 no

In [53]:
date_cols = ['CERTIFICATE-DATE', 'EXPIRATION-DATE']
for col in date_cols:
    if col in dealer_df.columns:
        dealer_df[col] = pd.to_datetime(dealer_df[col], format='%Y%m%d', errors='coerce')

In [54]:
dealer_df.columns = dealer_df.columns.str.strip()

df_obj_colds = dealer_df.select_dtypes(['object']).columns

# 3. Apply stripping only to the filtered list
dealer_df[df_obj_colds] = dealer_df[df_obj_colds].apply(lambda x: x.astype(str).str.strip())

# 4. Drop the ghost column if it exists
if 'Unnamed: 38' in dealer_df.columns:
    dealer_df = dealer_df.drop(columns=['Unnamed: 38'])

/var/folders/r5/7sdqfmvs4p987n2g0n_nng840000gn/T/ipykernel_88499/686333371.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  df_obj_colds = dealer_df.select_dtypes(['object']).columns


In [55]:
cols_to_fix = dealer_df.columns

dealer_df[cols_to_fix] = dealer_df[cols_to_fix].replace(r'^\s*$', np.nan, regex=True)

In [56]:
dealer_df.nunique()

CERTIFICATE-NUMBER         12400
OWNERSHIP                      7
CERTIFICATE-DATE            5100
EXPIRATION-DATE             5096
EXPIRATION-FLAG                1
CERTIFICATE-ISSUE-COUNT       41
NAME                       11035
STREET                      8391
STREET2                      278
CITY                        2824
STATE-ABBREV                  54
ZIP-CODE                    8191
OTHER-NAMES-COUNT              5
OTHER-NAMES-1               1450
OTHER-NAMES-2                130
OTHER-NAMES-3                 10
OTHER-NAMES-4                  1
OTHER-NAMES-5                  1
OTHER-NAMES-6                  1
OTHER-NAMES-7                  1
OTHER-NAMES-8                  1
OTHER-NAMES-9                  1
OTHER-NAMES-10                 1
OTHER-NAMES-11                 1
OTHER-NAMES-12                 1
OTHER-NAMES-13                 0
OTHER-NAMES-14                 0
OTHER-NAMES-15                 0
OTHER-NAMES-16                 0
OTHER-NAMES-17                 0
OTHER-NAME

## Cleaning `OWNERSHIP` column

From the FAA dataset description, we know:
* 1 - Individual
* 2 - Partnership
* 3 - Corporation
* 4 - Co-Ownership
* 7 - LLC
* 8 – Non Citizen Corporation

In [57]:
dealer_df['OWNERSHIP'].value_counts(dropna=False)

OWNERSHIP
3    4254
7    3648
1    2427
0    1813
2     150
4      95
8      13
Name: count, dtype: int64

In [58]:
ownership_map = {
    '1': 'Individual',
    '2': 'Partnership',
    '3': 'Corporation',
    '4': 'Co-Ownership',
    '7': 'LLC',
    '8': 'Non Citizen Corporation'
}

dealer_df['OWNERSHIP'] = dealer_df['OWNERSHIP'].astype(str).map(ownership_map)

In [59]:
dealer_df['OWNERSHIP'].value_counts(dropna=False)

OWNERSHIP
Corporation                4254
LLC                        3648
Individual                 2427
NaN                        1813
Partnership                 150
Co-Ownership                 95
Non Citizen Corporation      13
Name: count, dtype: int64

In [60]:
dealer_df

,CERTIFICATE-NUMBER,OWNERSHIP,CERTIFICATE-DATE,EXPIRATION-DATE,EXPIRATION-FLAG,CERTIFICATE-ISSUE-COUNT,NAME,STREET,STREET2,CITY,STATE-ABBREV,ZIP-CODE,OTHER-NAMES-COUNT,OTHER-NAMES-1,OTHER-NAMES-2,OTHER-NAMES-3,OTHER-NAMES-4,OTHER-NAMES-5,OTHER-NAMES-6,OTHER-NAMES-7,OTHER-NAMES-8,OTHER-NAMES-9,OTHER-NAMES-10,OTHER-NAMES-11,OTHER-NAMES-12,OTHER-NAMES-13,OTHER-NAMES-14,OTHER-NAMES-15,OTHER-NAMES-16,OTHER-NAMES-17,OTHER-NAMES-18,OTHER-NAMES-19,OTHER-NAMES-20,OTHER-NAMES-21,OTHER-NAMES-22,OTHER-NAMES-23,OTHER-NAMES-24,OTHER-NAMES-25
0,00-0027,Corporation,2000-01-04,2001-01-03,*,0,BIRD-EARLE JOINT VENTURES INC,4900 US HIGHWAY 1 N STE 300,NaN,SAINT AUGUSTINE,FL,320956265,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,00-0037,LLC,2000-01-05,2001-01-04,*,0,FLY AND BE FREE LLC,32 WALKER DR,NaN,SIMSBURY,CT,060702624,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,00-0040,Individual,2000-01-05,2001-01-04,*,0,CALL O JAY,83 POOL CREEK RD,NaN,LIVINGSTON,MT,590479117,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,00-0056,Corporation,2000-01-12,2001-01-11,*,0,MAC AIR CORP,PO BOX 189,NaN,MCPHERSON,KS,674600189,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,00-0060,Individual,2000-01-07,2001-01-06,*,0,CONNOLLY WILLIAM DBA,2520 TAILSPIN TRAIL,NaN,DAYTONA BEACH,FL,32124,1,SHAMROCK AIR-1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12395,DCN9965,NaN,NaT,NaT,NaN,0,PINKNEY TODD MICHAEL,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
12396,DCN9980,LLC,NaT,NaT,NaN,0,EPC AIRCRAFT SALES LLC,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
12397,DCN9988,NaN,NaT,NaT,NaN,0,WEAVER AERO INTERNATIONAL,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
12398,DCN9994,NaN,NaT,NaT,NaN,0,RWR AVIATION LLC,NaN,NaN,NaN,NaN,NaN,1,RASCH ROBERT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Cleaning `Expiration` column

In [61]:
dealer_df['EXPIRATION-FLAG'].value_counts(dropna=False)

EXPIRATION-FLAG
*      8592
NaN    3808
Name: count, dtype: int64

In [62]:
expiration_flag = {
    '*': 'true',
    np.nan: 'false'
}

dealer_df['EXPIRATION-FLAG'] = dealer_df['EXPIRATION-FLAG'].astype(str).map(expiration_flag)

In [63]:
dealer_df['EXPIRATION-FLAG'].value_counts(dropna=False)

EXPIRATION-FLAG
true     8592
false    3808
Name: count, dtype: int64

## Column Name Cleaning

In [64]:
dealer_df = dealer_df.rename(columns={
    'STATE-ABBREV': 'state',
    'EXPIRATION-FLAG': 'expired'
})

In [65]:
dealer_df.columns = (dealer_df.columns
                     .str.replace(' ', '_', regex=False)
                     .str.replace('-', '_', regex=False)
                     .str.replace('(', '', regex=False)
                     .str.replace(')', '', regex=False)
                     .str.lower())

In [66]:
dealer_df.to_csv('./clean_data/DEALER.csv', index=False, na_rep='')